In [ ]:
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
import random

# Directory containing the dataset
dataPath = r'C:\Users\baraa\OneDrive\Desktop\Traffic-Sign-Recognition-master\DataSet\images'

# Function to load images and labels
def loadDataSet(dataPath):
    data = [] # store images and their labels
    imageSize = 28
    for category in os.listdir(dataPath):
        label = int(category[:2]) # use the first two numbers of folder names as the label for the images (00, 01, 02, 03)
        for i in os.listdir(dataPath + '/' + category): # cd into each category folder
            imagePath = dataPath + '/' + category + '/'+ i # i is each image. (Dataset Folder -> category folders -> images in each folder)
            try:
                image = cv2.imread(imagePath,0)  # '0': grayscale mode
                image = cv2.resize(image,(imageSize,imageSize)) # resize to set pixels
                data.append([image,label]) # append list of processed image and its label to the 'data' list.
            except:
                pass
                
    random.shuffle(data) # shuffle the data to generalize and create unbiased dataset
    
    X = [] # store image
    Y = [] # store labels
    for img,lbls in data: # prepare images and labels for for model processing 
        X.append(img)
        Y.append(lbls)
    X = np.array(X)
    Y = np.array(Y).reshape(len(Y),1)
    return X,Y

#Load Data Set 
images,labels=loadDataSet(dataPath)

In [ ]:
#plot some examples from our images
for i in range(3): 
    plt.imshow(images[i],cmap='gray')
    plt.show()
    print(labels[i])

In [ ]:
# Mean and Standard Deviation , becouse we need it Later to Normalize the images 
mean=np.mean(images)
std=np.std(images)
print("The Mean=",mean)
print("The Standard Deviation=",std)

images.shape

In [ ]:
# prepare for training 
m=images.shape[0] # Store size of first dimention of images, which represents the total number of images.
imageShape=images.shape[1:] # other dimentions, which are width and height (32x32)
print(f'Total number of images: {m}')

In [ ]:
# Splitting Data into Training Set and Testing Set
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(images,labels,test_size=0.2,random_state=0)

# standarize the data to improve training process (normalizes training data)
x_trainNorm = (x_train - mean) / std 
x_testNorm  = (x_test - mean) / std
x_trainNorm = x_trainNorm.reshape(x_train.shape[0], 28, 28,1)
x_testNorm = x_testNorm.reshape(x_test.shape[0], 28, 28,1)

In [ ]:
# Adding Data Augmentation to introduce variatino in the traiing data. (Due to inaccurate detection in tests)

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Create an instance of ImageDataGenerator with augmentation parameters
data_gen = ImageDataGenerator(
    rotation_range=10,  # degrees
    width_shift_range=0.1,  # fraction of total width
    height_shift_range=0.1,  # fraction of total height
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [ ]:
# Processing Stage:
def preprocessingImage(image=None, imageSize=28, mean=mean, std=std):
    # must repeat the normalization preprocessing steps (color conversion, standardization, resizing)
    try:
        if image.shape[-1] == 3:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    except:
        pass
    image = cv2.resize(image, (imageSize, imageSize))
    image = (image - mean) / std
    image = image.reshape(1, imageSize, imageSize, 1)
    return image

In [ ]:
import tensorflow as tf
from tensorflow import keras
            
model = keras.Sequential([
    keras.layers.Conv2D(filters=16, kernel_size=(3,3), activation='relu', input_shape=(28,28,1)),
    keras.layers.MaxPool2D(pool_size=(2,2)),
    keras.layers.Conv2D(filters=32, kernel_size=(3,3), activation='relu'),
    keras.layers.MaxPool2D(pool_size=(2,2)),
    keras.layers.Dropout(0.25),  # Dropout layer
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),  # Increased Dense layer size
    keras.layers.Dropout(0.5),  # Another Dropout layer
    keras.layers.Dense(4, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# Fitting model
batch_size = 32  # You can adjust the batch size
train_generator = data_gen.flow(x_trainNorm, y_train, batch_size=batch_size)
steps_per_epoch = x_trainNorm.shape[0] // batch_size

model.fit(train_generator, 
          steps_per_epoch=steps_per_epoch, 
          epochs=10, 
          validation_data=(x_testNorm, y_test))

In [ ]:
model.evaluate(x_testNorm,y_test, verbose = 0)

In [ ]:
model.save('Models/TrafficDetectionModel')

In [ ]:
directory = r'C:\Users\baraa\OneDrive\Desktop\Traffic-Sign-Recognition-master\UnseenImages'

for filename in os.listdir(directory):
    imgPath = os.path.join(directory, filename)
    image = cv2.imread(imgPath, 1)

    if image is not None:
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))  # Display in RGB format
        plt.show()
        img = preprocessingImage(image)
        pred = model.predict(img) * 100

        print("Stop = " + str(pred[0][0]) + "%")
        print("Do not Enter = " + str(pred[0][1]) + "%")
        print("Traffic Jam = " + str(pred[0][2]) + "%")
        print("Yeild = " + str(pred[0][3]) + "%")